# Artwork Multi-Method Eval — KVzip & Finch (CPT vs Full)
Results + checkpoint saved to **Google Drive**.


## Step 0.5 — GitHub PAT

In [ ]:
import getpass, os
manual = getpass.getpass("GitHub Classic PAT (leave empty to use Colab Secret): ").strip()
if manual:
    gh_token = manual
else:
    try:
        from google.colab import userdata
        gh_token = (userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception:
        gh_token = os.environ.get("GITHUB_TOKEN", "").strip()
print("Token loaded:", bool(gh_token))

## Step 1 — Clone repos

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

BENCH_URL = "https://github.com/physical168/kv-compression-benchmark.git"
CE_URL    = "https://github.com/GabrieleSanmartino/CompressionExperiments.git"
BENCH_DIR = Path("/content/kv-compression-benchmark")
CE_DIR    = Path("/content/CompressionExperiments")

def clone(url, out, token, name):
    r = subprocess.run(["git","clone","--depth","1",url,str(out)], capture_output=True, text=True)
    if r.returncode == 0: print(name, "public clone OK"); return True
    print(name, "public clone failed"); print((r.stderr or "")[:300])
    if not token: return False
    r2 = subprocess.run(["git","clone","--depth","1",url.replace("https://github.com/",f"https://{token}@github.com/"),str(out)], capture_output=True, text=True)
    if r2.returncode == 0: print(name, "token clone OK"); return True
    print(name, "token clone failed"); print((r2.stderr or "")[:300]); return False

_tok = (globals().get("gh_token") or "").strip() or None
os.chdir("/content")
for p in [BENCH_DIR, CE_DIR]:
    if p.exists(): shutil.rmtree(p)
clone(BENCH_URL, BENCH_DIR, _tok, "BENCH")
clone(CE_URL, CE_DIR, _tok, "CE")


## Step 1.2 — Install dependencies

In [ ]:
import subprocess, sys, os
from pathlib import Path
_ce = Path("/content/CompressionExperiments")

subprocess.check_call([sys.executable,"-m","pip","install","-q","-U","pip"])
subprocess.check_call([sys.executable,"-m","pip","install","-q",
    "transformers>=5.0","accelerate","bitsandbytes","pandas","pyyaml","scikit-learn","matplotlib","tqdm","pillow"])

# Install kvpress
_kvp = _ce / "kvpress"
has_setup = (_kvp / "setup.py").is_file() or (_kvp / "pyproject.toml").is_file()
installed_ok = False
if _kvp.is_dir() and has_setup:
    print("Installing kvpress from CE submodule...")
    try:
        subprocess.check_call([sys.executable,"-m","pip","install",str(_kvp)])
        installed_ok = True
    except: pass

if not installed_ok:
    print("Installing kvpress from PyPI...")
    subprocess.check_call([sys.executable,"-m","pip","install","kvpress"])

import kvpress
print("kvpress version:", getattr(kvpress, "__version__", "unknown"))


## Step 1.3 — Hotfix kvpress (Fix FinchPress Hook Bug)

In [ ]:
import kvpress, os, inspect
from pathlib import Path

# Fix FinchPress 'hook' variable bug
finch_path = Path(inspect.getfile(kvpress.FinchPress))
if finch_path.is_file():
    txt = finch_path.read_text()
    # 1. Ensure hook is initialized to None in __enter__
    if 'hook = None' not in txt:
        txt = txt.replace('def __enter__(self, model):', 'def __enter__(self, model):\n        self.hook = None')
    # 2. Use self.hook in remove check
    txt = txt.replace('hook.remove()', 'if hasattr(self, "hook") and self.hook: self.hook.remove()')
    finch_path.write_text(txt)
    print(f"Patched {finch_path.name}")

# Fix BasePress tested_models warning spam
base_path = Path(inspect.getfile(kvpress.presses.base_press.BasePress))
if base_path.is_file():
    txt = base_path.read_text()
    txt = txt.replace('logger.warning(f"Model {type(model)} not tested', '# logger.warning(f"Model {type(model)} not tested')
    base_path.write_text(txt)
    print(f"Patched {base_path.name}")


## Step 1.5 — Mount Google Drive

In [ ]:
from google.colab import drive
import os, pandas as pd
from pathlib import Path
drive.mount("/content/drive", force_remount=False)
DRIVE_OUT        = "/content/drive/MyDrive/kv-compression-benchmark/ce_artwork_all_results"
Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH  = Path(DRIVE_OUT) / "checkpoint_all.csv"
RESULTS_ROOT     = Path(DRIVE_OUT) / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)


## Step 2 — Prepare dataset (Robust Download)

In [ ]:
import os, shutil, urllib.request, time
from pathlib import Path
from urllib.parse import unquote
import pandas as pd

CE_ART = Path("/content/CompressionExperiments/experiment_manager/datasets/artwork")
CE_ART.mkdir(parents=True, exist_ok=True)
IMAGES_DIR = CE_ART / "images"
IMAGES_DIR.mkdir(exist_ok=True)
DATASET_PATH = CE_ART / "paintings.csv"

if not DATASET_PATH.is_file():
    shutil.copy2("/content/kv-compression-benchmark/paintings.csv", DATASET_PATH)

def is_valid_img(p):
    return p.is_file() and p.stat().st_size > 1000

df_csv = pd.read_csv(DATASET_PATH)
# We only need the first few for the benchmark (MAX_ROWS is usually 10)
MAX_TO_DOWNLOAD = 15 
target_urls = df_csv["image_url"].dropna().head(MAX_TO_DOWNLOAD)

# Using a more descriptive User-Agent to be nice to Wikimedia
headers = {"User-Agent": "ArtworkBenchmarkBot/1.0 (contact: github.com/physical168)"}

print(f"Checking first {MAX_TO_DOWNLOAD} images...")
for url in target_urls:
    fname_unquoted = unquote(url.split("/")[-1].split("?")[0])
    fname_quoted = url.split("/")[-1].split("?")[0]
    
    # Check if either quoted or unquoted version exists and is valid
    dst = IMAGES_DIR / fname_unquoted
    if is_valid_img(dst) or is_valid_img(IMAGES_DIR / fname_quoted):
        continue 

    # If exists but invalid (LFS pointer), delete it
    if dst.exists(): dst.unlink()
    
    for attempt in range(3):
        try:
            print(f"Downloading (Attempt {attempt+1}): {fname_unquoted}...")
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=20) as r, open(dst, "wb") as f:
                f.write(r.read())
            print(f"✅ Success: {fname_unquoted}")
            time.sleep(2) # Rate limit
            break
        except Exception as e:
            print(f"❌ Fail: {fname_unquoted} - {e}")
            time.sleep(5)

print(f"Images ready at: {IMAGES_DIR} | count: {len(list(IMAGES_DIR.glob('*')))}")


## Step 2.5 — Pre-flight Check (Verify Before Model Load)

In [ ]:
import pandas as pd
from pathlib import Path
from urllib.parse import unquote

# This cell ensures you don't waste time loading the model if data is missing
DATASET_PATH = "/content/CompressionExperiments/experiment_manager/datasets/artwork/paintings.csv"
IMAGES_DIR = Path("/content/CompressionExperiments/experiment_manager/datasets/artwork/images")
MAX_ROWS = globals().get("MAX_ROWS", 10)

df = pd.read_csv(DATASET_PATH).head(MAX_ROWS)
missing_or_bad = []

print(f"Validating the first {MAX_ROWS} images...")
for i, row in df.iterrows():
    url = row["image_url"]
    fn_un = unquote(url.split("/")[-1].split("?")[0])
    fn_qu = url.split("/")[-1].split("?")[0]
    
    p_un, p_qu = IMAGES_DIR / fn_un, IMAGES_DIR / fn_qu
    
    target = None
    if p_un.is_file() and p_un.stat().st_size > 1000: target = p_un
    elif p_qu.is_file() and p_qu.stat().st_size > 1000: target = p_qu
    
    if target:
        print(f"✅ Row {i}: {target.name} OK")
    else:
        print(f"❌ Row {i}: MISSING/CORRUPT")
        missing_or_bad.append(i)

if not missing_or_bad:
    print("\n🚀 ALL IMAGES READY! Proceed to load model.")
else:
    raise FileNotFoundError(f"🛑 {len(missing_or_bad)} images missing. Re-run Step 2.")


## Step 3 — Load model

In [ ]:
import sys, importlib.util, torch, transformers
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration, BitsAndBytesConfig

MODEL_ID  = "llava-hf/llama3-llava-next-8b-hf"
dtype     = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
bnb_cfg   = BitsAndBytesConfig(load_in_8bit=True, llm_int8_enable_fp32_cpu_offload=True)

processor = LlavaNextProcessor.from_pretrained(MODEL_ID)
model = LlavaNextForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map="auto", quantization_config=bnb_cfg)

_patch = "/content/kv-compression-benchmark/benchmarks/artwork_eval/llava_kvpress_patch.py"
_spec = importlib.util.spec_from_file_location("patch", _patch)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
_mod.apply_kvpress_compatibility_patches(model)
print("Model ready.")


## Step 4 — Configuration

In [ ]:
from kvpress import ExpectedAttentionPress, KVzipPress, FinchPress

MAX_ROWS = 10
MAX_NEW_TOKENS = 50
COMPRESSION_RATIOS = [0.4, 0.8, 0.95]

EXPERIMENT_CONFIGS = {
    "KVzip": (KVzipPress, True),
    "Finch_Full": (FinchPress, False),
    "Finch_CPT": (FinchPress, True),
}

_EXTRACT_SUFFIX = (" (be concise, no explanation, no introductory text, just the answer,"
                   " output datatype: STRING, do not repeat the datatype in the answer) ?")

def build_answer_prefix(row, question: str, is_boolean: bool, use_cpt: bool) -> str:
    context = "" if use_cpt else f"This painting is titled '{row['title']}', created around {row['inception']}. It belongs to the {row['movement']} movement and {row['genre']} genre. "
    if is_boolean:
        return (f"{context}Answer the following question based on the image with '1' or '0'."
                f" Do not add any other comments. {question}\nAnswer: ")
    return (f"{context}Answer the following question based on the image."
            f" Do not add any other comments. {question + _EXTRACT_SUFFIX}\nAnswer: ")


## Step 5 — Load dataset & queries

In [ ]:
import yaml, pandas as pd
from urllib.parse import unquote
with open("/content/kv-compression-benchmark/benchmarks/artwork_eval/configs/image_queries.yaml") as f:
    q_cfg = yaml.safe_load(f)["artwork"]
queries_plan = [(q, True) for q in q_cfg["filter_queries"]] + [(q, False) for q in q_cfg["extract_queries"]]

df = pd.read_csv(DATASET_PATH).head(MAX_ROWS)
df["record_id"] = range(len(df))
def resolve_image(url: str) -> str:
    tail = url.split("/")[-1].split("?")[0]
    p = IMAGES_DIR / unquote(tail)
    return str(p) if p.is_file() else str(IMAGES_DIR / tail)
df["image_path"] = df["image_url"].apply(resolve_image)


## Step 6 — Multi-Method Inference

In [ ]:
import csv as _csv, gc, os, time, torch, logging
from PIL import Image

# Silence redundant KVzip warnings
logging.getLogger("kvpress.presses.kvzip_press").setLevel(logging.ERROR)

_tok = getattr(processor, "tokenizer", processor)

def run_generate_vision(image_path, answer_prefix, PressCls, ratio):
    press = PressCls(compression_ratio=float(ratio))
    if hasattr(press, "update_model_and_tokenizer"):
        press.update_model_and_tokenizer(model, _tok)
    image = Image.open(image_path).convert("RGB")
    conv  = [{"role":"user","content":[{"type":"image"},{"type":"text","text":answer_prefix}]}]
    prompt = processor.apply_chat_template(conv, add_generation_prompt=True)
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device)
    gen_kw = {"max_new_tokens": MAX_NEW_TOKENS, "pad_token_id": _tok.pad_token_id}
    with torch.no_grad(), press(model):
        out = model.generate(**inputs, **gen_kw)
    return _tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def load_done(path):
    if not os.path.exists(path): return set()
    ck = pd.read_csv(path)
    done = set()
    for _, r in ck.iterrows():
        if str(r.get("error","")).strip().lower() in ("","nan"):
            done.add((str(r["method"]), str(float(r["ratio"])), int(r["record_id"]), str(r["query"])))
    return done

done_keys = load_done(CHECKPOINT_PATH)
print(f"Resuming: {len(done_keys)} already done.")

f = open(CHECKPOINT_PATH, "a", newline="", encoding="utf-8")
writer = _csv.DictWriter(f, fieldnames=["method","ratio","record_id","query","answer","error"])
if os.path.getsize(CHECKPOINT_PATH) == 0:
    writer.writeheader()

total_tasks = len(EXPERIMENT_CONFIGS) * len(COMPRESSION_RATIOS) * len(df) * len(queries_plan)
n_step = 0
n_err = 0

for m_name, (Cls, use_cpt) in EXPERIMENT_CONFIGS.items():
    print(f"\n>>> Starting Method: {m_name}")
    for ratio in COMPRESSION_RATIOS:
        for _, row in df.iterrows():
            for qtext, is_bool in queries_plan:
                n_step += 1
                key = (m_name, str(float(ratio)), int(row["record_id"]), qtext)
                if key in done_keys: continue
                
                ap = build_answer_prefix(row, qtext, is_bool, use_cpt)
                ans = err = ""
                try:
                    ans = run_generate_vision(row["image_path"], ap, Cls, ratio)
                except Exception as e:
                    err = str(e)[:500]
                    n_err += 1
                    print(f"❌ ERROR [rid={row.record_id} {m_name} R{ratio}]: {err[:100]}")
                
                writer.writerow({"method":m_name,"ratio":ratio,"record_id":row.record_id,"query":qtext,"answer":ans,"error":err})
                f.flush()
                
                if n_step % 20 == 0:
                    print(f"Progress: {n_step}/{total_tasks} | Errors so far: {n_err}")
                gc.collect(); torch.cuda.empty_cache()

f.close()
print(f"\nFinished! Total Errors: {n_err}")


## Step 7-8 — Merge & Evaluate

In [ ]:
import pandas as pd
from pathlib import Path
ck_all = pd.read_csv(CHECKPOINT_PATH)
OLD_EA_PATH = Path("/content/drive/MyDrive/kv-compression-benchmark/ce_artwork_results/checkpoint.csv")
if OLD_EA_PATH.is_file():
    ea_df = pd.read_csv(OLD_EA_PATH).rename(columns={'press': 'method'})
    ea_df['method'] = 'EA'
    ck_combined = pd.concat([ea_df, ck_all], ignore_index=True).drop_duplicates(subset=['method','ratio','record_id','query'], keep='first')
else:
    ck_combined = ck_all

ck_ok = ck_combined[ck_combined.error.fillna("").astype(str).str.strip().isin(["","nan"])].copy()
for (m, r), grp in ck_ok.groupby(["method", "ratio"]):
    d = RESULTS_ROOT / "artwork" / f"Llava3-8b-{m}" / f"{float(r):.2f}"
    d.mkdir(parents=True, exist_ok=True)
    grp[["record_id","query","answer"]].to_csv(d / "results.csv", index=False)

import sys
sys.path.append("/content/kv-compression-benchmark/benchmarks/artwork_eval")
from evaluation.evaluator import EvaluationManager
mgr = EvaluationManager(config_path="/content/kv-compression-benchmark/benchmarks/artwork_eval/evaluation/evaluation_config.yaml", results_dir=str(RESULTS_ROOT))
res = mgr.evaluate_all()
display(res.groupby(["model_tag","ratio"])[["precision","recall","f1"]].mean())
